## 2.1 图像预处理- 统一输入格式

#### 1. 为什么需要图像预处理

##### 1.1 什么是图像预处理
图像预处理（Image Preprocessing）指的是：

在把图像送入模型之前，对图像进行一系列整理、转换和优化操作。
* 让模型更容易学习
* 提高训练稳定性
* 提高模型泛化能力

📌 也就是说，预处理服务的对象主要不是人，而是模型。🤖

##### 1.2 为什么不能直接把原图扔进 CNN
真实世界中的图片往往存在很多问题：
* 尺寸不一致
* 通道格式不一致
* 像素范围不一致
* 图像质量不一致
* 训练集样本不足
* 不同图片拍摄角度、亮度、位置不同
如果直接把这些“原始状态”的图像送进模型，模型学习起来会更困难，效果也可能不稳定。
所以我们通常要先做预处理，把图像整理成更适合训练的形式。

##### 1.3 图像在深度学习里的“流转路线”
在真正做 CNN 时，一张图像通常不会一直保持同一种格式，而是会在不同对象之间来回转换：
* 磁盘中的图片文件：`.jpg / .png`
* matplotlib 读出来后：通常是 `NumPy ndarray`
* 送入 PyTorch 前：要变成 `torch.Tensor`
* 进入模型训练前：通常还要做
```
    * 类型转换
    * 归一化
    * 通道顺序调整
    * 批量维度添加
```

所以这一节的核心不是“形状变化本身”，而是理解：
* 图像在 NumPy 和 Tensor 之间是怎么转换的
* 转换时有哪些坑 ⚠️
* 数据类型、数值范围、内存共享分别有什么影响

##### 1.4 图像预处理的两个大目标
图像预处理里的操作，可以先粗略分成两类：

**（1）格式整理类**
这类操作是为了让图像能顺利进入模型，例如：
* resize
* 转 Tensor
* dtype 转换
* 归一化
* 标准化
* 补通道
* 调整维度顺序

**（2）数据增强类**

这类操作是为了让模型学得更强、更稳，例如：
* 随机裁剪
* 随机翻转
* 随机旋转
* 颜色抖动
* 加噪声

📌 前者偏“让它能用”，后者偏“让它更强”。

#### 2. 图像在 NumPy 和 PyTorch 中分别是什么

##### 2.1 NumPy 中的图像
在 Python 里，图像经常先以 `NumPy ndarray` 的形式存在。

例如：
``` python
import matplotlib.image as mpimg
img = mpimg.imread("cat.png")
print(type(img))
print(img.shape)
print(img.dtype)
```
可能输出：
```
<class 'numpy.ndarray'>
(300, 400, 3)
float32
```
这说明：
* 图像本质是 ndarray
* 它是一个多维数组
* 每个元素就是像素值

📌 所以：

NumPy 眼里的图像 = 一个数组

##### 2.2 PyTorch 中的图像
在 PyTorch 中，图像通常要变成 `torch.Tensor`

例如：
``` python
import torch

tensor_img = torch.tensor(img)
print(type(tensor_img))
print(tensor_img.shape)
print(tensor_img.dtype)
```
可能输出：
```
<class 'torch.Tensor'>
torch.Size([300, 400, 3])
torch.float32
```
📌 所以：

PyTorch 眼里的图像 = 一个张量

##### 2.3 它们本质上的共同点
其实 NumPy ndarray 和 torch.Tensor 很像：
* 都能表示多维数据
* 都有 shape
* 都有 dtype
* 都支持索引、切片、数学运算

区别在于：
* NumPy 更偏向通用科学计算
* PyTorch 更偏向深度学习和自动求导
* PyTorch 可以方便地放到 GPU 上训练 🚀

##### ⚠️ 存在的问题：
1. dtype问题：
    * numpy的ndarray可能读取到的dtype是 uint8
    * CNN模型能接受的dtype为float32
2. 大小问题:
    * uint8 的大小范围是 0-255
    * CNN 最佳的接受范围是经过归一化处理 0-1
3. 维度数量问题：
    * 对于灰度图像转换后的shape为 H x W
        * 但是CNN模型中需要接受通道信息
        * 所以需要进行unsqueeze（）处理增加通道维度
    * CNN模型训练也需要接受batch数据，所以也需要增加一个batch维度
4. 维度顺序问题：
    * pytorch中维度顺序通常是 C 排在最后
    * 但是CNN模型接受的数据 C 应该排在 batch维度之后

#### 3. NumPy 图像转 PyTorch Tensor

##### 3.1 最常见的两种方式
**（1）torch.tensor() 深拷贝 ✅**

In [1]:
import numpy as np
import torch

img_np = np.array([[0, 128], [200, 255]], dtype=np.uint8)
img_tensor = torch.tensor(img_np)

print("Numpy array type:", img_np.dtype)
print("PyTorch tensor type:", img_tensor.dtype)

Numpy array type: uint8
PyTorch tensor type: torch.uint8


**（2）torch.from_numpy() 浅拷贝 🙅**

In [2]:
img_tensor_2 = torch.from_numpy(img_np)
print("PyTorch tensor from numpy type:", img_tensor_2.dtype)

PyTorch tensor from numpy type: torch.uint8


##### 3.2 torch.tensor() 和 torch.from_numpy() 的区别
这是很重要的点。⭐
```
方式	            是否共享内存	是否常用于图像预处理
torch.tensor(arr)	    否	        可以
torch.from_numpy(arr)	是	        很常用
```

In [4]:
arr = np.array([1,2,3,4,5], dtype=np.float32)
t1 = torch.tensor(arr)
t2 = torch.from_numpy(arr)

arr[0] = 10
print("Numpy array:", arr)
print("PyTorch tensor t1:", t1)
print("PyTorch tensor t2:", t2)

Numpy array: [10.  2.  3.  4.  5.]
PyTorch tensor t1: tensor([1., 2., 3., 4., 5.])
PyTorch tensor t2: tensor([10.,  2.,  3.,  4.,  5.])


说明：
* t1 是拷贝，不受影响
* t2 和原数组共用内存，跟着变了

📌 结论：
* 想高效转换，常用 from_numpy()
* 不想互相影响，可以用 tensor()

#### 4. PyTorch Tensor 转回 NumPy

##### 4.1 基本转换方式

In [6]:
t = torch.tensor([1,2,3,4,5], dtype=torch.float32)

arr = t.numpy()
print("dtype of PyTorch tensor:", t.dtype)
print("Numpy array from PyTorch tensor:", arr)

dtype of PyTorch tensor: torch.float32
Numpy array from PyTorch tensor: [1. 2. 3. 4. 5.]


##### 4.2 Tensor 转 NumPy 的前提
如果 Tensor 在 CPU 上，通常可以直接：

`arr = t.numpy()`

但是如果 Tensor 在 GPU 上，就不能直接转，需要先搬回 CPU：

In [8]:
arr_2 = t.cpu().numpy()
print("Numpy array from PyTorch tensor on CPU:", arr_2)

Numpy array from PyTorch tensor on CPU: [1. 2. 3. 4. 5.]


📌 后面训练 CNN 时非常常见：
* 模型在 GPU 上跑
* 想拿结果画图或打印
* 就必须先 .cpu() 再 .numpy()

##### 4.3 为什么有时还会看到 .detach().cpu().numpy()
后面训练中你会经常看到：

In [9]:
arr_3 = t.detach().cpu().numpy()
print("Numpy array from detached PyTorch tensor on CPU:", arr_3)

Numpy array from detached PyTorch tensor on CPU: [1. 2. 3. 4. 5.]


含义：
* detach()：从计算图中分离
* cpu()：搬回 CPU
* numpy()：转为 NumPy 数组

这通常用于：
* 可视化预测结果
* 画 loss 曲线
* 打印中间输出

📌 当前阶段先记住：

如果 Tensor 参与梯度计算，转 NumPy 前常常要先 detach()。

#### 5. 图像的 dtype 和归一化（重点）💡

##### 5.1 为什么 dtype 很重要
同样是一张图像，像素内容可能一样，但 dtype 不同，含义可能不一样。

常见情况：
* uint8：像素范围通常是 0 ~ 255
* float32：可能是 0 ~ 1
* float64：数学计算常见，但深度学习里不常作为图像输入

``` python
import numpy as np

img1 = np.array([[0, 128, 255]], dtype=np.uint8)
img2 = np.array([[0.0, 0.5, 1.0]], dtype=np.float32)
```
它们看起来表达的亮度类似，但数值范围不同。

##### 5.2 为什么模型通常更喜欢 float32
因为神经网络训练时：
* 参数通常是 float32
* 梯度计算通常也基于 float32
* 这样速度和内存更平衡

所以图像输入前通常会写：

`img_tensor = img_tensor.float()`

##### 5.3 最常见的图像输入预处理
**（1）原图是 uint8，范围 0~255**

`img_tensor = torch.from_numpy(img_np).float() / 255.0`

这一步做了两件事：
* 转成 float32
* 归一化到 0~1

**（2）原图已经是 float32，范围 0~1**

有些读取方式读出来已经是这样了，那就不需要再除以 255。

所以实际工作中一定要先检查：
```
print(img_np.dtype)
print(img_np.min(), img_np.max())
```
📌 这是非常好的习惯。✅


#### 6. 图像shape（重点）💡

##### 6.1 为什么图像转换不仅仅是“格式变了”
对于图像来说，最麻烦的问题通常不是数组转 Tensor，而是：

`通道顺序不一致。`

在前面的学习中我们已经知道：
* matplotlib / NumPy 中图像常见格式：(H, W, C)
* PyTorch 模型更常见输入格式：(C, H, W)

所以图像从 NumPy 进 PyTorch 时，经常要调整通道位置。

##### 6.2  灰度图虽然是二维，但在 CNN 中通常要明确成单通道
灰度图在 NumPy 中通常长这样：

`(H, W)`

但是在 CNN 看来，它其实应该表示为：

`(1, H, W)`

也就是：
* 1 个通道
* 高度 H
* 宽度 W

所以对于单通道图像，我们常常需要手动增加一个通道维度。

##### 6.3 单通道通过 unsqueeze() 增加通道维度
例如一张灰度图：
``` python
import torch
import numpy as np

gray_np = np.random.randint(0, 256, size=(28, 28), dtype=np.uint8)
gray_tensor = torch.from_numpy(gray_np).float() / 255.0

print(gray_tensor.shape)
```
输出：

`torch.Size([28, 28])`

这时它只有两个维度，还不能直接理解为标准的 CNN 单张图像输入。

我们可以用 unsqueeze() 增加一个通道维度：
``` python
gray_tensor = gray_tensor.unsqueeze(0)
print(gray_tensor.shape)
```
输出：

`torch.Size([1, 28, 28])`

这里的 0 表示：

在第 0 维位置新增一个维度。

📌 所以：
* 原来：(H, W)
* 增加通道后：(1, H, W)

这就明确表示：这是一张单通道图像。

##### 6.4  如果后面还要组成 batch，还要再增加一个 batch 维度
例如：
``` python
gray_tensor = gray_tensor.unsqueeze(0)
print(gray_tensor.shape)
```
如果前一步已经是 (1, 28, 28)，那这一步之后会变成：

`torch.Size([1, 1, 28, 28])`

这就表示：
* batch size = 1
* channel = 1
* height = 28
* width = 28

📌 所以对于灰度图，常见过程是：
* (H, W)
* unsqueeze(0) → (1, H, W)
* 再 unsqueeze(0) → (1, 1, H, W)

第一个新增的是通道维度，第二个新增的是批量维度。

##### 6.5 RGB 图像天然有通道，但顺序常常不对
RGB 图像在 NumPy / matplotlib 中通常是：

`(H, W, 3)`

但 PyTorch 的 CNN 层通常希望输入的是：

`(C, H, W)`

所以这里不是“增加通道”，而是要调整通道位置。

##### 6.6 CNN中的维度顺序和常见图像读取顺序不一致
这里要特别注意一个概念：
* 图像读取库（如 matplotlib、很多 NumPy 场景）更常见：HWC
* PyTorch CNN 更常见：CHW
* PyTorch 批量输入 更常见：NCHW

所以同一张图像，在不同场景中看到的维度顺序可能不同。

##### 6.7 通过 transpose() 或 permute() 转换维度顺序
**（1）NumPy 中常用 transpose()**
``` python
import numpy as np

rgb_np = np.random.randint(0, 256, size=(224, 224, 3), dtype=np.uint8)
print(rgb_np.shape)
```
输出：

`(224, 224, 3)`

如果想把它从 HWC 变成 CHW，可以这样做：
``` python
rgb_np_chw = rgb_np.transpose(2, 0, 1)
print(rgb_np_chw.shape)
```

输出：

`(3, 224, 224)`

这里的 (2, 0, 1) 表示：
* 原来的第 2 维（通道）放到最前面
* 原来的第 0 维（高）放到第二位
* 原来的第 1 维（宽）放到第三位

**（2）PyTorch 中常用 permute()**

如果已经是 Tensor：
``` python
import torch

rgb_tensor = torch.from_numpy(rgb_np).float() / 255.0
print(rgb_tensor.shape)
```

输出：

`torch.Size([224, 224, 3])`

这时可以使用 permute()：
``` python 
rgb_tensor = rgb_tensor.permute(2, 0, 1)
print(rgb_tensor.shape)
```

输出：

`torch.Size([3, 224, 224])`

📌 permute(2, 0, 1) 的含义和刚才 transpose(2, 0, 1) 的思路一致，都是把维度顺序重排成：
* C
* H
* W

#### 7. 一个完整例子：从 RGB NumPy 图像到 PyTorch CNN 可用格式

In [16]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
# 1. 读取图片
img_arr = mpimg.imread('/home/zhang/Documents/Programming/DataAnalysis_AI-Learning/3.DL_Learning/4. CNN/data/26 (19).jpg')

# 2. 检查图片arr的属性，包括dtype，shape，min，max
print("Image array dtype:", img_arr.dtype)
print("Image array shape:", img_arr.shape)
print("Pixel value range:", np.min(img_arr), "to", np.max(img_arr))

# 3. 转换为 PyTorch tensor + dtype + 归一化
# 3.1 直接转换为 float32
img_tensor = torch.tensor(img_arr).float() # 直接转换为 float32
# 3.2 归一化到 [0, 1]
img_tensor /= 255.0
print("Processed image tensor dtype:", img_tensor.dtype)
print("Processed pixel value range:", torch.min(img_tensor).item(), "to", torch.max(img_tensor).item())

# 4. 针对维度个数和顺序进行处理
# 本例图片是一个灰度图片，shape 是 (H, W)，我们需要增加一个 C 维度，变成 (1，H, W)，表示单通道并且符合CNN的输入要求。
# 4.1 增加一个 C 通道信息
img_tensor = img_tensor.unsqueeze(0) # 在 ndim=0 的位置增加一个维度，变成 (1, H, W, C)
# 4.2 新增一个batch维度，变成 (B, C, H, W)，其中 B=1
img_tensor = img_tensor.unsqueeze(0) # 在 ndim=0 的位置增加一个维度，变成 (1, 1, H, W)

# 5. 检查最后的 tensor 的 dtype 和 shape
print("Final image tensor dtype: ", img_tensor.dtype)
print("Final image tensor shape: ", img_tensor.shape)
print("Final image tensor pixel value range:", torch.min(img_tensor).item(), "to", torch.max(img_tensor).item())


Image array dtype: uint8
Image array shape: (208, 176)
Pixel value range: 0 to 254
Processed image tensor dtype: torch.float32
Processed pixel value range: 0.0 to 0.9960784316062927
Final image tensor dtype:  torch.float32
Final image tensor shape:  torch.Size([1, 1, 208, 176])
Final image tensor pixel value range: 0.0 to 0.9960784316062927


##### 🧠 总结：通道处理规律
对于灰度图：
* NumPy 常见：(H, W)
* CNN 单张输入常见：(1, H, W)
* CNN 批量输入常见：(N, 1, H, W)

对于 RGB 图：
* NumPy 常见：(H, W, 3)
* CNN 单张输入常见：(3, H, W)
* CNN 批量输入常见：(N, 3, H, W)

处理手段：
* 灰度图补通道：unsqueeze()
* 维度顺序转换：transpose() / permute()
* 增加 batch：unsqueeze()

#### 8. 图像转换中的常见坑⚠️

##### 8.1 坑一：以为转换成功就代表可以训练
`不一定。`

例如：

`img_tensor = torch.from_numpy(img_np)`

虽然已经是 Tensor，但可能仍然有问题：
* dtype 不对
* 通道顺序不对
* 数值范围不对
* 缺少 batch 维度

##### 8.2 坑二：忘记检查范围
比如有的图像已经是 0~1，你又除以 255，会导致数值非常小，影响训练。
所以一定先检查：
```
print(img.dtype)
print(img.min(), img.max())
```

##### 8.3 坑三：共享内存带来的“联动修改”
用 torch.from_numpy() 时：
* 改 NumPy，Tensor 可能变
* 改 Tensor，NumPy 也可能变

调试时有时会让人很困惑。

##### 8.4 坑四：GPU Tensor 不能直接 .numpy()
错误示意：
`arr = gpu_tensor.numpy()`

正确思路：

`arr = gpu_tensor.cpu().numpy()`

##### 8.5 坑五：把“显示格式”和“训练格式”混为一谈
这是图像学习前期最常见的问题之一。

你要始终问自己：
* 我现在是为了显示？
* 还是为了送进模型？